# Principal Component Analysis (PCA)

**INDE 577 / CMOR 438 — Rice University**  
**Instructor:** Randy R. Davila, PhD

---

## Overview

PCA is an **unsupervised dimensionality reduction** technique that finds the directions of **maximum variance** in the data. It projects the data onto a lower-dimensional subspace while preserving as much variance as possible.

## Mathematical Background

### Covariance Matrix

Given centered data $\tilde{\mathbf{X}} = \mathbf{X} - \bar{\mathbf{x}}$:

$$\mathbf{\Sigma} = \frac{1}{n-1} \tilde{\mathbf{X}}^T \tilde{\mathbf{X}} \in \mathbb{R}^{d \times d}$$

### Eigendecomposition

$$\mathbf{\Sigma} \mathbf{v}_j = \lambda_j \mathbf{v}_j$$

The **principal components** are the eigenvectors $\mathbf{v}_1, \mathbf{v}_2, \ldots$, sorted by decreasing eigenvalue $\lambda_1 \geq \lambda_2 \geq \ldots$

### Projection

Project the data onto the top $k$ principal components:

$$\mathbf{Z} = \tilde{\mathbf{X}} \mathbf{V}_k$$

where $\mathbf{V}_k = [\mathbf{v}_1, \ldots, \mathbf{v}_k] \in \mathbb{R}^{d \times k}$.

### Explained Variance Ratio

The proportion of total variance explained by the $j$-th principal component:

$$\text{EVR}_j = \frac{\lambda_j}{\sum_{i=1}^{d} \lambda_i}$$

### Reconstruction (Inverse Transform)

$$\hat{\mathbf{X}} = \mathbf{Z} \mathbf{V}_k^T + \bar{\mathbf{x}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_breast_cancer
from sklearn.decomposition import PCA as SklearnPCA
import warnings
warnings.filterwarnings('ignore')

from rice_ml import PCA
from rice_ml.processing.preprocessing import StandardScaler

print("Libraries loaded!")
np.random.seed(42)

## 1. Understanding PCA on Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

# Full PCA
pca_full = PCA(n_components=4)
X_pca_full = pca_full.fit_transform(X_s)

print("=== PCA on Iris ===")
print(f"Original shape:  {X.shape}")
print(f"Reduced shape:   {X_pca_full.shape}")
print()
print("Explained Variance Ratio per Component:")
for i, evr in enumerate(pca_full.explained_variance_ratio_):
    print(f"  PC{i+1}: {evr:.4f} ({evr:.1%})")
print(f"\nCumulative EVR: {np.cumsum(pca_full.explained_variance_ratio_)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']

# Scree plot
ax = axes[0]
evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)
ax.bar(range(1, 5), evr, color='steelblue', edgecolor='k', lw=0.5, alpha=0.8, label='Individual')
ax.step(range(1, 5), cum_evr, color='crimson', linewidth=2.5, where='mid', label='Cumulative')
ax.scatter(range(1, 5), cum_evr, color='crimson', s=60, zorder=3)
ax.axhline(0.95, color='gray', linestyle='--', linewidth=1.5, label='95% threshold')
ax.set_xlabel('Principal Component', fontsize=11)
ax.set_ylabel('Explained Variance Ratio', fontsize=11)
ax.set_title('Scree Plot — Iris', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# 2D projection
ax2 = axes[1]
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_s)
for i, (name, color) in enumerate(zip(target_names, colors)):
    mask = y == i
    ax2.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, s=50, alpha=0.8,
                edgecolors='k', lw=0.4, label=name)
ax2.set_xlabel(f'PC1 ({evr[0]:.1%} var)', fontsize=11)
ax2.set_ylabel(f'PC2 ({evr[1]:.1%} var)', fontsize=11)
ax2.set_title('Iris Projected onto PC1-PC2', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Loading plot (PC1 vs PC2 components)
ax3 = axes[2]
components = pca_2d.components_
for j, (fname, c1, c2) in enumerate(zip(feature_names, components[0], components[1])):
    ax3.arrow(0, 0, c1, c2, head_width=0.03, head_length=0.02, fc='steelblue', ec='steelblue', lw=1.5)
    offset = np.array([c1, c2]) * 1.15
    ax3.text(offset[0], offset[1], fname[:12], fontsize=9, ha='center', va='center',
             color='darkblue', fontweight='bold')
ax3.axhline(0, color='gray', lw=0.8)
ax3.axvline(0, color='gray', lw=0.8)
circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', alpha=0.5)
ax3.add_patch(circle)
ax3.set_xlim(-1.3, 1.3)
ax3.set_ylim(-1.3, 1.3)
ax3.set_xlabel('PC1 Loadings', fontsize=11)
ax3.set_ylabel('PC2 Loadings', fontsize=11)
ax3.set_title('PCA Biplot (Loading Vectors)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')

plt.tight_layout()
plt.savefig('pca_iris.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Image Compression with PCA (Digits Dataset)

In [ ]:
digits = load_digits()
X_dig = digits.data.astype(float)  # 1797 × 64

# Standardize
scaler_dig = StandardScaler()
X_dig_s = scaler_dig.fit_transform(X_dig)

# Show reconstruction quality for different # components
n_components_list = [2, 10, 30, 64]

fig, axes = plt.subplots(len(n_components_list) + 1, 10, figsize=(12, 8))

# Original
for j in range(10):
    axes[0, j].imshow(X_dig[j].reshape(8, 8), cmap='gray_r')
    axes[0, j].axis('off')
    if j == 0:
        axes[0, j].set_ylabel('Original', fontsize=9, rotation=0, labelpad=40, ha='right', va='center')

for row, n_comp in enumerate(n_components_list):
    pca_d = PCA(n_components=n_comp)
    X_proj = pca_d.fit_transform(X_dig_s)
    X_recon = pca_d.inverse_transform(X_proj)
    X_recon_orig = scaler_dig.inverse_transform(X_recon)
    
    evr_sum = pca_d.explained_variance_ratio_.sum()
    
    for j in range(10):
        axes[row + 1, j].imshow(X_recon_orig[j].reshape(8, 8), cmap='gray_r')
        axes[row + 1, j].axis('off')
    if True:
        axes[row + 1, 0].set_ylabel(f'k={n_comp}\n({evr_sum:.0%})',
                                     fontsize=9, rotation=0, labelpad=45, ha='right', va='center')

plt.suptitle('PCA Image Reconstruction (Digits Dataset)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('pca_digits_reconstruction.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cumulative explained variance for digits
pca_all = PCA(n_components=64)
pca_all.fit(X_dig_s)
cum_evr = np.cumsum(pca_all.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(1, 65), cum_evr, 'steelblue', linewidth=2)
for threshold in [0.90, 0.95, 0.99]:
    n_comp = np.argmax(cum_evr >= threshold) + 1
    ax.axhline(threshold, color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax.axvline(n_comp, color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax.text(n_comp + 0.5, threshold - 0.02, f'{threshold:.0%}: {n_comp} PCs',
            fontsize=9, color='red')
ax.set_xlabel('Number of Components', fontsize=11)
ax.set_ylabel('Cumulative Explained Variance', fontsize=11)
ax.set_title('Cumulative Variance — Digits Dataset', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# 2D visualization of digits with PCA
ax2 = axes[1]
pca_2d_dig = PCA(n_components=2)
X_dig_2d = pca_2d_dig.fit_transform(X_dig_s)
scatter = ax2.scatter(X_dig_2d[:, 0], X_dig_2d[:, 1], c=digits.target,
                      cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, ax=ax2, ticks=range(10), label='Digit')
ax2.set_xlabel(f'PC1 ({pca_2d_dig.explained_variance_ratio_[0]:.1%})', fontsize=11)
ax2.set_ylabel(f'PC2 ({pca_2d_dig.explained_variance_ratio_[1]:.1%})', fontsize=11)
ax2.set_title('Digits Projected to 2D PCA', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pca_digits_2d.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. PCA as Preprocessing for Classification

In [ ]:
from rice_ml import LogisticRegression
from rice_ml.processing.preprocessing import train_test_split
from rice_ml.processing.metrics import accuracy_score

bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target

scaler_bc = StandardScaler()
X_bc_s = scaler_bc.fit_transform(X_bc)
X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(X_bc_s, y_bc, test_size=0.2, random_state=42)

results = []
n_comp_range = [2, 5, 10, 15, 20, 25, 30]

# Baseline: no PCA
lr_base = LogisticRegression(learning_rate=0.05, n_iterations=500)
lr_base.fit(X_tr_bc, y_tr_bc)
base_acc = accuracy_score(y_te_bc, lr_base.predict(X_te_bc))
print(f"Baseline (30 features): {base_acc:.4f}")

for n_comp in n_comp_range:
    pca_p = PCA(n_components=n_comp)
    X_tr_pca = pca_p.fit_transform(X_tr_bc)
    X_te_pca = pca_p.transform(X_te_bc)
    
    lr = LogisticRegression(learning_rate=0.05, n_iterations=500)
    lr.fit(X_tr_pca, y_tr_bc)
    acc = accuracy_score(y_te_bc, lr.predict(X_te_pca))
    evr_sum = pca_p.explained_variance_ratio_.sum()
    results.append((n_comp, acc, evr_sum))
    print(f"PCA({n_comp:2d} comps, {evr_sum:.1%} var): {acc:.4f}")

plt.figure(figsize=(9, 4))
n_comps, accs, evrs = zip(*results)
plt.plot(n_comps, accs, 'go-', linewidth=2, markersize=6, label='PCA + LR')
plt.axhline(base_acc, color='red', linestyle='--', linewidth=2, label=f'No PCA ({base_acc:.3f})')
plt.xlabel('Number of PCA Components', fontsize=11)
plt.ylabel('Test Accuracy', fontsize=11)
plt.title('PCA as Preprocessing: Accuracy vs Dimensionality', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_classification.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Property | Value |
|---|---|
| Type | Unsupervised linear dimensionality reduction |
| Method | Eigendecomposition of covariance matrix |
| Output | Projection onto top $k$ principal components |
| Preserves | Maximum variance |
| Applications | Visualization, noise reduction, preprocessing, compression |

**Key Takeaways:**
- PCA finds directions of **maximum variance** — not maximum class separation (that's LDA)
- Always **standardize features** before PCA, otherwise high-variance features dominate
- Use the **scree plot / elbow method** to choose the number of components
- PCA can be used as preprocessing to reduce noise and speed up downstream algorithms
- The first 2 PCs can be used to **visualize** high-dimensional data